In [1]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import pandas as pd

# Import du processeur de production spécialisé
from tools.OI_class_OP import OI_ProductionProcessor
from tools.OI_Dashboard import ProductionDashboard
from tools.OI_Dashboard_v2 import AjouterVisualisationsAvancees


# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_columns', None)

In [2]:
# Cellule 2 : Définition des métadonnées de tags API
tags = [
    {'tag':'WQ33222VA', 'nom':'ester_cons','info':'Totalisation du Peson Acetate/Propionate', 'agg': 'FIRST' },
    {'tag':'NOP_ESTERS', 'nom':'ester_nop','info':"Nombres des opérations d'esters",'agg':'FIRST' },
    {'tag':'3340_type', 'nom':'A/P','info':'Acetate ou Propionate','agg':'FIRST' },
    {'tag':'CTY_ACV43A_Teneur Vit. A (UV)', 'nom':'Acetate_UV','info':'ACV43A teneur en acetate', 'agg': 'MEAN'},
    {'tag':'CTY_A3340FGB_Teneur arr. Vit. A (UV)', 'nom':'Propionate_UV', 'info':'A3340FGB teneur en propionate', 'agg': 'MEAN'},
    {'tag':'PU3310VA_Sign','nom':'PU3310','info':'signature du PU3310','agg':'FIRST'},
    {'tag':'PU3320VA_Sign','nom':'PU3320','info':'signature du PU3320','agg':'FIRST'},
    {'tag':'PU3340VA_Sign','nom':'PU3340','info':'signature du PU3340','agg':'FIRST'},
    {'tag':'FQ32202VA_UV','nom':'Hexane','info':'VA diluée dans de l\'hexane', 'agg': 'FIRST'},
    {'tag':'CTY_A3230A_Teneur en rétinol', 'nom':'Retinol_UV','info':'A3230A teneur en rétinol lavé', 'agg': 'MEAN'},
    {'tag':'LI33203VA','nom':'R33020','info':'niveau du R33020', 'agg': 'FIRST'},
    {'tag':'LI33218VA','nom':'R33022','info':'niveau du R33022', 'agg': 'FIRST'},
    {'tag':'LI33225VA','nom':'R33061','info':'niveau du R33061', 'agg': 'FIRST'},
    {'tag':'WI33222VA','nom':'R33060','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'FQ32202VA','nom':'retinol_cons','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'LI32209VA','nom':'R32031','info':'niveau du R32031', 'agg': 'FIRST'},
]

In [3]:
# Cellule 3 : Variable Produits unifiée (Acetate & Propionate)
# Plus aucune distinction batch / continu pour le calcul global du stock d'un produit.
produits = [
    {
        'nom': 'Ester',
        'conso': {
            'value': 'ester_cons',
            'scale': 1e-9,
            'type':'A/P',
            'uv': ['Acetate_UV', 'Propionate_UV'],
        },
        'CMJ': 9.5,
        'NOP': {
            'value':'NOP_ESTERS',
            'scale': 1,
            'type': None,
        },
        'stock': [
            # Batchs
            {'pu': 'PU3310', 'in': 540, 'out': 710, 'value': 'Hexane', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
            {'pu': 'PU3310', 'in': 710, 'out': 2320, 'value': None, 'uv': None, 'scale': 2.71},
            {'pu': 'PU3320', 'in': 430, 'out': 2020, 'value': None, 'uv': None, 'scale': 2.71},
            # Continus
            {'pu': None, 'value': 'R33020', 'min': 14, 'epalage': [[28.62, 21.22, 3.866, -0.0983], [-228.84, 74.557]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33022', 'min': 14, 'epalage': [[6.25, 5.4525, 0.898, -0.0256], [-35, 97, 16.039]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33061', 'min': 18, 'epalage': [[0.69, 0.72, 0.296, -0.0059], [-30.03, 5.836]], 'uv': None, 'scale': 0.95 , 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': 'PU3340', 'in': 0, 'out': 710, 'value': 'R33060', 'uv': None, 'scale': 0.95, 'cond': 'A/P', 'val_cond': [1/344, 1/359]}
        ]
    },
    {
        'nom': 'Retinol',
        'conso': {
            'value':'retinol_cons',
            'scale': 1e-5, # Ajustement de l'échelle pour le retinol g et analyse en %
            'uv': ['Retinol_UV','Retinol_UV'],
        },
        'CMJ': 1,
        'stock': [
            # Batchs
            {'pu': None, 'value': 'R32031', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
        ]
    }
]

In [4]:
# Cellule 4 : Initialisation du processeur de production spécialisé
processor = OI_ProductionProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start = '2026-01-01',
    end = '2026-12-31',
    tags_metadata = tags,
    produits = produits,
    interval = 'PT15M',
    verbose = False
)

In [5]:
(processor.agg_mapping)

{'WQ33222VA': 'FIRST',
 'NOP_ESTERS': 'FIRST',
 '3340_type': 'FIRST',
 'CTY_ACV43A_Teneur Vit. A (UV)': 'MEAN',
 'CTY_A3340FGB_Teneur arr. Vit. A (UV)': 'MEAN',
 'PU3310VA_Sign': 'FIRST',
 'PU3320VA_Sign': 'FIRST',
 'PU3340VA_Sign': 'FIRST',
 'FQ32202VA_UV': 'FIRST',
 'CTY_A3230A_Teneur en rétinol': 'MEAN',
 'LI33203VA': 'FIRST',
 'LI33218VA': 'FIRST',
 'LI33225VA': 'FIRST',
 'WI33222VA': 'FIRST',
 'FQ32202VA': 'FIRST',
 'LI32209VA': 'FIRST'}

In [6]:
# Cellule 5 : Téléchargement et calcul automatique des bilans par produit
processor.merge()
processor.compute_production_balance()

# Visualisation des premières lignes calculées
processor.data.describe()

,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester_tmp_PU3310_Hexane_0,stock_Ester_tmp_PU3310_None_1,stock_Ester_tmp_PU3320_None_2,stock_Ester_tmp_None_R33020_3,stock_Ester_tmp_None_R33022_4,stock_Ester_tmp_None_R33061_5,stock_Ester_tmp_PU3340_R33060_6,stock_Ester,consommation_Retinol,stock_Retinol_tmp_None_R32031_0,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
count,1.688300e+04,16883.000000,16883.000000,1.688300e+04,1.688300e+04,16875.000000,16877.000000,16877.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,1.688300e+04,16883.000000,16883.000000,16883.000000,16883.000000,16883.000000,1.688300e+04,16883.000000,1.688300e+04
mean,2.493194e+06,1996.543602,0.065623,2.226376e+06,2.268292e+06,1036.030222,961.650945,460.761391,2621.457752,20.758015,46.391532,5.145974,7.411168,692.128142,507800.204854,48.018720,606.456581,0.029968,1.426510,1.593929,1.943876,0.034981,0.078476,1.427386,6.535126,4.910823e+05,0.012080,0.012080,606.456581,0.948838,607.405419,4.910823e+05,0.000525,4.910823e+05
std,1.711371e+05,140.268077,0.247587,3.647053e+04,5.745005e+04,618.092540,553.857401,119.137623,1875.635978,0.354717,20.340863,2.507960,11.507851,350.819230,277146.412908,21.117573,383.278828,0.145838,1.353152,1.333808,0.910154,0.016742,0.168193,0.948850,2.127870,5.259819e+05,0.005313,0.005313,383.278828,2.127870,383.842891,5.259819e+05,0.005313,5.259819e+05
min,2.221310e+06,1774.000000,0.000000,2.078915e+06,2.188000e+06,100.000000,100.000000,100.000000,0.000000,20.000000,-8.142390,-0.695313,-0.418438,4.689820,640.456000,-1.303840,0.000000,0.000000,0.000000,0.000000,0.000805,0.001665,0.001218,0.000000,0.707269,0.000000e+00,-0.000329,-0.000329,0.000000,-4.879019,-1.628659,0.000000e+00,-0.011884,0.000000e+00
25%,2.332040e+06,1865.000000,0.000000,2.221983e+06,2.242000e+06,400.000000,400.000000,410.000000,0.000000,20.600000,24.788000,3.778570,2.309895,435.814000,270043.000000,39.671250,244.107751,0.000000,0.000000,0.000000,0.975332,0.022909,0.010631,0.699863,5.418261,7.839144e+01,0.009955,0.009955,244.107751,-0.168027,246.214634,7.839144e+01,-0.001600,7.839283e+01
50%,2.475520e+06,1984.000000,0.000000,2.231549e+06,2.242000e+06,1310.000000,1110.000000,410.000000,3959.170000,20.700000,46.276100,6.050780,3.187500,681.504000,487835.000000,45.130700,566.204966,0.000000,2.710000,2.710000,1.930145,0.039986,0.016009,1.507355,6.847844,1.783350e+02,0.011523,0.011523,566.204966,1.261557,570.339879,1.783350e+02,-0.000032,1.783322e+02
75%,2.641180e+06,2117.000000,0.000000,2.241926e+06,2.314000e+06,1310.000000,1555.000000,410.000000,4042.610000,21.200000,65.167000,6.683570,4.460265,950.604500,752052.000000,67.177200,938.088508,0.000000,2.710000,2.710000,2.782455,0.045304,0.025514,2.197955,7.979079,9.992032e+05,0.016848,0.016848,938.088508,2.392792,939.963621,9.992032e+05,0.005293,9.992032e+05
max,2.804730e+06,2251.000000,1.000000,2.325113e+06,2.626000e+06,2340.000000,2250.000000,810.000000,4836.070000,21.200000,91.590400,12.284600,73.884600,1323.150000,999821.000000,79.734400,1304.435574,1.207552,2.710000,2.710000,3.933059,0.099400,1.107856,3.590475,12.323393,1.997291e+06,0.019909,0.019909,1304.435574,6.737105,1308.215394,1.997291e+06,0.008354,1.997291e+06


In [7]:
# ✨ INITIALISER LE DASHBOARD ✨
dashboard = ProductionDashboard(processor)
AjouterVisualisationsAvancees(dashboard) 

print("\n✓ Dashboard prêt pour utilisation!")
#dashboard.resume_complet()


✓ Dashboard initialisé
  Produits: Ester, Retinol
  Période: 2026-01-01 → 2026-06-26
✅ Visualisations avancées ajoutées au dashboard!

   Nouvelles méthodes disponibles:
   • dashboard.plot_histogramme_tous_produits(mois=3)
   • dashboard.plot_waterfall_mois(mois=3)
   • dashboard.plot_histogramme_jours_mois_v1(mois=3, nom_produit='Ester')
   • dashboard.plot_histogramme_jours_mois_v2(mois=3, nom_produit='Ester')


✓ Dashboard prêt pour utilisation!


In [8]:
# Voir l'évolution temporelle d'un produit
dashboard.afficher_bilan_produit('Ester')


In [16]:
dashboard.plot_histogramme_jours_mois_v2(mois=6, annea=2026, nom_produit='Ester', std=2)

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.3908
  Variation de Stock     : 0.0057
  Stock Entrée.          : -0.0004
  Stock Sortie.          : 0.0052
  Production             : 3.3964
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JUIN 2026 (Termin


  📊 STATISTIQUES - PRODUCTION ESTER PAR JOUR - JUNE 2026
Nombre de jours complets:           26
Production moyenne:                 8.20
Production min/max:                 2.59 / 14.13
Écart-type:                         2.71
Coefficient de variation:           33.0%
Production totale mois:             213.29
TRS mois: (jours complets)          8.20
Jours au-dessus de la moyenne:      21 / 26

🎯 RATIO oee (Production / CMJ en %):
  CMJ (Cible Journalière):            9.50
  oee Moyen (par jour):               86.4%
  oee Min/Max (par jour):             27.3% / 148.7%
  oee Cumulé à date:                  86.4%
  Jours > 100%:                       7 / 26

  Détail oee par jour:
    J01:  113.6% ✅
    J02:   85.6% ⚠️ 
    J03:   94.0% ⚠️ 
    J04:  113.8% ✅
    J05:   77.9% ❌
    J06:  148.7% ✅
    J07:   99.1% ⚠️ 
    J08:   74.7% ❌
    J09:  104.3% ✅
    J10:   86.2% ⚠️ 
    J11:   27.3% ❌
    J12:   88.5% ⚠️ 
    J13:   89.8% ⚠️ 
    J14:  116.4% ✅
    J15:   88.6% ⚠️ 
    J16:  11

In [10]:
debut = '06-01-2026 02:00:00'
fin = '06-02-2026 02:00:00'
NOP = processor.data.loc[[fin],['ester_nop']].values[0] - processor.data.loc[[debut],['ester_nop']].values[0]
print('Ester NOP   : ', NOP, NOP * 2.71)
print('Ester peson : ',processor.data.loc[[fin],['ester_cons']].values[0] - processor.data.loc[[debut],['ester_cons']].values[0] )
print('Ester titre : ',processor.data.loc[[fin],['consommation_Ester']].values[0] - processor.data.loc[[debut],['consommation_Ester']].values[0])
print('Ester Stock : ',processor.data.loc[[fin],['stock_Ester']].values[0], processor.data.loc[[debut],['stock_Ester']].values[0] )
print('Ester Delta : ',processor.data.loc[[fin],['stock_Ester']].values[0] - processor.data.loc[[debut],['stock_Ester']].values[0])
print('Ester conso : ',processor.data.loc[[fin],['conso_delta_Ester']].values[0]- processor.data.loc[[debut],['conso_delta_Ester']].values[0] )
print('Ester prod  : ',processor.data.loc[[fin],['production_Ester']].values[0]- processor.data.loc[[debut],['production_Ester']].values[0] )
processor.data[debut:fin]

Ester NOP   :  [4.] [10.84]
Ester peson :  [3740.]
Ester titre :  [8.3173024]
Ester Stock :  [7.76989323] [5.29724975]
Ester Delta :  [2.47264348]
Ester conso :  [8.3173024]
Ester prod  :  [10.78994588]


,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester_tmp_PU3310_Hexane_0,stock_Ester_tmp_PU3310_None_1,stock_Ester_tmp_PU3320_None_2,stock_Ester_tmp_None_R33020_3,stock_Ester_tmp_None_R33022_4,stock_Ester_tmp_None_R33061_5,stock_Ester_tmp_PU3340_R33060_6,stock_Ester,consommation_Retinol,stock_Retinol_tmp_None_R32031_0,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-06-01 02:00:00+00:00,2710770.0,2175.00,0.0,2231260.0,2188000.0,400.0,1120.0,710.0,0.00,20.6,57.9978,6.94141,10.02970,1256.3900,717187.0,44.5753,1095.154465,0.0,0.00,2.71,2.452419,0.047191,0.087640,0.000000,5.297250,999249.829855,0.011130,0.011130,1095.154465,-0.289038,1094.865427,999249.829855,-0.000425,999249.829430
2026-06-01 02:15:00+00:00,2710770.0,2175.00,0.0,2231260.0,2188000.0,400.0,1120.0,710.0,0.00,20.6,57.2839,6.91016,15.94610,1252.7800,717187.0,44.8255,1095.154465,0.0,0.00,2.71,2.420545,0.046925,0.175403,0.000000,5.352874,999249.829855,0.011193,0.011193,1095.154465,-0.233414,1094.921051,999249.829855,-0.000362,999249.829493
2026-06-01 02:30:00+00:00,2710770.0,2175.00,0.0,2231260.0,2188000.0,400.0,1135.0,810.0,0.00,20.6,55.6637,6.91016,22.43230,1251.0400,717187.0,72.0132,1095.154465,0.0,0.00,2.71,2.348207,0.046925,0.278607,0.000000,5.383739,999249.829855,0.017981,0.017981,1095.154465,-0.202548,1094.951917,999249.829855,0.006426,999249.836282
2026-06-01 02:45:00+00:00,2710770.0,2175.92,0.0,2231260.0,2188000.0,400.0,1200.0,410.0,0.00,20.6,53.4612,6.87500,29.39710,36.5625,717187.0,70.5189,1095.154465,0.0,0.00,2.71,2.249871,0.046626,0.390857,0.100972,5.498327,999249.829855,0.017608,0.017608,1095.154465,-0.087960,1095.066505,999249.829855,0.006053,999249.835908
2026-06-01 03:00:00+00:00,2710770.0,2176.00,0.0,2231260.0,2188000.0,400.0,1110.0,410.0,0.00,20.6,51.1893,6.87500,3.64990,214.3000,717187.0,71.4949,1095.154465,0.0,0.00,2.71,2.148437,0.046626,0.019260,0.591817,5.516140,999249.829855,0.017852,0.017852,1095.154465,-0.070147,1095.084318,999249.829855,0.006297,999249.836152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-02 01:00:00+00:00,2714510.0,2178.00,0.0,2220216.0,2188000.0,1310.0,1610.0,810.0,3975.14,20.6,45.8798,3.36135,22.32030,348.8450,733647.0,17.8650,1103.471767,0.0,2.71,2.71,1.911381,0.020212,0.276801,0.000000,7.628394,999253.220615,0.004461,0.004461,1103.471767,2.042107,1105.513874,999253.220615,-0.007094,999253.213521
2026-06-02 01:15:00+00:00,2714510.0,2179.00,0.0,2220216.0,2188000.0,1310.0,1620.0,410.0,3975.14,20.6,43.5132,3.25881,3.34044,167.6370,733647.0,16.5130,1103.471767,0.0,2.71,2.71,1.805718,0.019564,0.017062,0.462951,7.725294,999253.220615,0.004123,0.004123,1103.471767,2.139007,1105.610774,999253.220615,-0.007432,999253.213183
2026-06-02 01:30:00+00:00,2714510.0,2179.00,0.0,2220216.0,2188000.0,1310.0,1620.0,410.0,3975.14,20.6,40.5426,3.18842,3.91220,216.4220,733647.0,57.9776,1103.471767,0.0,2.71,2.71,1.673088,0.019123,0.021220,0.597677,7.731108,999253.220615,0.014477,0.014477,1103.471767,2.144821,1105.616588,999253.220615,0.002922,999253.223537


In [11]:
processor.calcul_cumul_journalier(1,6,2026,"Ester")

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
____________________________________________________________


{'Ester': {'consommation': 8.317302400000017,
  'delta_stock': 2.472643477741661,
  'production': 10.78994587774173}}

In [12]:
dashboard.afficher_bilan_journalier(jour=6, mois=6, annee=2026)

____________________________________________________________
BILAN JOURNALIER - 06 JUIN 2026 (Terminé)
Période : du 06/06/2026 à 02:00 au 07/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 9.5224
  Variation de Stock     : 4.6081
  Stock Entrée.          : 1.1661
  Stock Sortie.          : 5.7741
  Production             : 14.1305
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.4445
  Variation de Stock     : 0.0018
  Stock Entrée.          : -0.0022
  Stock Sortie.          : -0.0003
  Production             : 3.4464
____________________________________________________________
____________________________________________________________


In [17]:
dashboard.plot_histogramme_annuee(annea=2026, nom_produit='Ester')

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -4.1001
  Stock Entrée.          : 3.0225
  Stock Sortie.          : -1.0776
  Production             : 4.1672
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Mois         Jours    Prod Total      Prod Moy        oee Moy      oee Cumul   
----------------------------------------------------------------------------------------------------
January      29       182.41          6.29            66.2        % 66.2        %
February     26       188.52          7.25            76.3        % 76.3        %
March        28       210.44          7.52            79.1        % 79.1        %
April        30       246.96          8.23            86.7        % 86.7        %
May          31       261.33          8.43            88.7        % 88.7        %
June         26       213.29          8.20            86.4        % 86.4        %



In [14]:
dashboard.plot_histogramme_annee_complet(annea=2026, nom_produit='Ester', std=2)

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -4.1001
  Stock Entrée.          : 3.0225
  Stock Sortie.          : -1.0776
  Production             : 4.1672
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Nombre de jours avec données:       170/365
Production totale année:            1302.95
Production moyenne (jours actifs):  7.66
Production min/max:                 0.03 / 14.13
Écart-type:                         2.95

🎯 RATIO oee:
  CMJ (Cible Journalière):            9.50
  oee Moyen (par jour):               80.7%
  oee Cumulé à date (fin d'année):    37.6%
  Jours > 100% (surproduction):       39 / 170



In [15]:
processor.plot_simple_tag('Hexane')